In [0]:
%sql
CREATE CATALOG IF NOT EXISTS global_weather_catalog;

In [0]:
%sql
USE CATALOG global_weather_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
%sql
USE CATALOG global_weather_catalog;
USE SCHEMA silver;

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM global_weather_catalog.bronze.bronze_weather;

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, current_timestamp, lit, to_timestamp, to_date,
    hour, month, year, dayofweek, when, trim, initcap, upper,
    abs as spark_abs, round as spark_round, mean, sum as spark_sum
)
import datetime
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_weather")

BUCKET      = "s3://global-weather-pipeline"
CATALOG     = "global_weather_catalog"
SILVER_PATH = f"{BUCKET}/silver/"
batch_date  = datetime.date.today().strftime("%Y-%m-%d")

# ── Read from Bronze ───────────────────────────────────────
df = spark.table(f"{CATALOG}.bronze.bronze_weather")
source_count = df.count()

print(f"✅ Bronze data loaded successfully!")
print(f"   Total rows    : {source_count}")
print(f"   Total columns : {len(df.columns)}")
print(f"\n📋 Columns:")
for c in df.columns:
    print(f"   {c}")

In [0]:
# ── Error Handling & Input Validation ─────────────────────
# This cell runs BEFORE any transformations.
# It validates the Bronze data coming in and raises clear errors
# if something is wrong — so we don't waste time transforming bad data.

import sys

errors   = []
warnings = []

# ── Check 1: Source has data ───────────────────────────────
if source_count == 0:
    errors.append("❌ Bronze table is empty — no records to process")

# ── Check 2: Minimum row threshold ────────────────────────
elif source_count < 1000:
    warnings.append(f"⚠️  Very few records in Bronze : {source_count} — expected 100,000+")

# ── Check 3: Required columns exist ───────────────────────
required_columns = [
    "weather_record_id", "country", "location_name",
    "last_updated", "temperature_celsius", "humidity",
    "wind_kph", "pressure_mb", "uv_index",
    "air_quality_pm2_5", "air_quality_pm10",
    "air_quality_us_epa_index", "air_quality_gb_defra_index"
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    errors.append(f"❌ Missing required columns : {missing_cols}")

# ── Check 4: Primary key must not be all null ──────────────
null_pk = df.filter(col("weather_record_id").isNull()).count()
if null_pk == source_count:
    errors.append(f"❌ weather_record_id is null for ALL records — PK column missing")
elif null_pk > 0:
    warnings.append(f"⚠️  {null_pk} null weather_record_ids found")

# ── Check 5: Critical numeric columns exist ────────────────
zero_temp = df.filter(
    col("temperature_celsius").isNull()
).count()
null_pct = round((zero_temp / source_count) * 100, 2)
if null_pct > 50:
    errors.append(f"❌ temperature_celsius is {null_pct}% null — data quality too low to proceed")
elif null_pct > 10:
    warnings.append(f"⚠️  temperature_celsius is {null_pct}% null — higher than expected")

# ── Check 6: Schema column count sanity check ──────────────
if len(df.columns) < 50:
    errors.append(f"❌ Only {len(df.columns)} columns found — expected 58. Schema mismatch from Bronze.")

# ── Print warnings ─────────────────────────────────────────
if warnings:
    print("⚠️  WARNINGS:")
    for w in warnings:
        print(f"   {w}")

# ── Raise errors if any ────────────────────────────────────
if errors:
    print("\n🚨 ERRORS FOUND — Silver ingestion cannot proceed:")
    for e in errors:
        print(f"   {e}")
    raise ValueError(f"Silver ingestion blocked due to {len(errors)} error(s). Fix Bronze data first.")

# ── All checks passed ──────────────────────────────────────
print("✅ All validation checks passed!")
print(f"   Check 1 — Source has data          : {source_count} rows ✅")
print(f"   Check 2 — Row count above threshold : {source_count} > 1000 ✅")
print(f"   Check 3 — All required columns exist: {len(required_columns)} columns found ✅")
print(f"   Check 4 — Primary key not null      : {null_pk} null PKs ✅")
print(f"   Check 5 — Temperature null %        : {null_pct}% ✅")
print(f"   Check 6 — Column count valid        : {len(df.columns)} columns ✅")
print(f"\n🟢 Silver ingestion is safe to proceed!")

In [0]:
# ── Data type standardization ──────────────────────────────
before_types = {f.name: str(f.dataType) for f in df.schema.fields}

df = (
    df
    .withColumn("weather_record_id",            col("weather_record_id").cast(IntegerType()))
    .withColumn("latitude",                     col("latitude").cast(DoubleType()))
    .withColumn("longitude",                    col("longitude").cast(DoubleType()))
    .withColumn("last_updated",                 to_timestamp("last_updated"))
    .withColumn("temperature_celsius",          col("temperature_celsius").cast(DoubleType()))
    .withColumn("feels_like_celsius",           col("feels_like_celsius").cast(DoubleType()))
    .withColumn("wind_kph",                     col("wind_kph").cast(DoubleType()))
    .withColumn("wind_degree",                  col("wind_degree").cast(IntegerType()))
    .withColumn("pressure_mb",                  col("pressure_mb").cast(DoubleType()))
    .withColumn("precip_mm",                    col("precip_mm").cast(DoubleType()))
    .withColumn("humidity",                     col("humidity").cast(IntegerType()))
    .withColumn("cloud",                        col("cloud").cast(IntegerType()))
    .withColumn("visibility_km",                col("visibility_km").cast(DoubleType()))
    .withColumn("uv_index",                     col("uv_index").cast(DoubleType()))
    .withColumn("gust_kph",                     col("gust_kph").cast(DoubleType()))
    .withColumn("air_quality_pm2_5",            col("air_quality_pm2_5").cast(DoubleType()))
    .withColumn("air_quality_pm10",             col("air_quality_pm10").cast(DoubleType()))
    .withColumn("air_quality_carbon_monoxide",  col("air_quality_carbon_monoxide").cast(DoubleType()))
    .withColumn("air_quality_ozone",            col("air_quality_ozone").cast(DoubleType()))
    .withColumn("air_quality_nitrogen_dioxide", col("air_quality_nitrogen_dioxide").cast(DoubleType()))
    .withColumn("air_quality_sulphur_dioxide",  col("air_quality_sulphur_dioxide").cast(DoubleType()))
    .withColumn("air_quality_us_epa_index",     col("air_quality_us_epa_index").cast(IntegerType()))
    .withColumn("air_quality_gb_defra_index",   col("air_quality_gb_defra_index").cast(IntegerType()))
    .withColumn("moon_illumination",            col("moon_illumination").cast(IntegerType()))
)

after_types = {f.name: str(f.dataType) for f in df.schema.fields}

# Show what changed
changed = {k: (before_types[k], after_types[k]) 
           for k in before_types 
           if before_types[k] != after_types[k]}

print(f"✅ Data type standardization complete!")
print(f"   Columns recast : {len(changed)}")
print(f"\n📊 Type changes:")
for col_name, (old, new) in changed.items():
    print(f"   {col_name}: {old} → {new}")

In [0]:
# ── String standardization ─────────────────────────────────
string_cols = ["country", "location_name", "timezone", "condition_text", "moon_phase"]

# Check before
print("📊 Before standardization (sample values):")
for c in string_cols:
    sample = df.select(c).filter(col(c).isNotNull()).first()[0]
    print(f"   {c}: '{sample}'")

# Apply initcap + trim
for c in string_cols:
    df = df.withColumn(c, initcap(trim(col(c))))

# wind_direction → uppercase
df = df.withColumn("wind_direction", upper(trim(col("wind_direction"))))

# Check after
print(f"\n📊 After standardization (sample values):")
for c in string_cols:
    sample = df.select(c).filter(col(c).isNotNull()).first()[0]
    print(f"   {c}: '{sample}'")

print(f"\n✅ String standardization complete!")
print(f"   Columns standardized (initcap + trim) : {len(string_cols)}")
print(f"   Columns standardized (upper)          : 1 — wind_direction")
print(f"   Total columns affected                : {len(string_cols) + 1}")

In [0]:
# ── Derived columns ────────────────────────────────────────
cols_before = len(df.columns)

df = (
    df
    .withColumn("date",        to_date("last_updated"))
    .withColumn("hour",        hour("last_updated"))
    .withColumn("month",       month("last_updated"))
    .withColumn("year",        year("last_updated"))
    .withColumn("day_of_week", dayofweek("last_updated"))
    .withColumn("is_day",      when(hour("last_updated").between(6, 18), "Day").otherwise("Night"))
)

cols_after = len(df.columns)

# Show sample
sample = df.select("last_updated", "date", "hour", "month", "year", "day_of_week", "is_day").limit(5)
display(sample)

print(f"\n✅ Derived columns added successfully!")
print(f"   Columns before : {cols_before}")
print(f"   Columns after  : {cols_after}")
print(f"   New columns added ({cols_after - cols_before}):")
print(f"      date        → extracted from last_updated")
print(f"      hour        → extracted from last_updated")
print(f"      month       → extracted from last_updated")
print(f"      year        → extracted from last_updated")
print(f"      day_of_week → extracted from last_updated (1=Sun, 7=Sat)")
print(f"      is_day      → 'Day' if hour 6-18, else 'Night'")

In [0]:
# ── Temperature validation ─────────────────────────────────
# Verifies temp_f = (temp_c × 9/5) + 32 within ±0.5 tolerance
# Catches sensor errors where celsius and fahrenheit don't match

from pyspark.sql.functions import abs as spark_abs

df = df.withColumn(
    "_temp_expected_f",
    spark_round(col("temperature_celsius") * 9.0 / 5.0 + 32, 1)
)

mismatch_count = df.filter(
    spark_abs(col("temperature_fahrenheit") - col("_temp_expected_f")) > 0.5
).count()

# Drop helper column
df = df.drop("_temp_expected_f")

if mismatch_count > 0:
    print(f"⚠️  Temperature mismatches found : {mismatch_count}")
else:
    print(f"✅ Temperature validation passed!")
    print(f"   All temperature_celsius values correctly match temperature_fahrenheit")
    print(f"   Mismatches found : {mismatch_count}")

In [0]:
# ── Null handling ──────────────────────────────────────────

# Check nulls BEFORE filling
print("📊 Null counts BEFORE handling:")
null_before = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).collect()[0].asDict()

total_nulls_before = 0
for k, v in null_before.items():
    if v > 0:
        print(f"   ⚠️  {k}: {v} nulls")
        total_nulls_before += v

print(f"\n   Total null values found : {total_nulls_before}")

# Calculate means from non-null rows only
temp_mean = df.filter(col("temperature_celsius").isNotNull()).select(mean("temperature_celsius")).first()[0]
hum_mean  = df.filter(col("humidity").isNotNull()).select(mean("humidity")).first()[0]
wind_mean = df.filter(col("wind_kph").isNotNull()).select(mean("wind_kph")).first()[0]
pm25_mean = df.filter(col("air_quality_pm2_5").isNotNull()).select(mean("air_quality_pm2_5")).first()[0]

# Fill nulls
df = df.fillna({
    "temperature_celsius" : float(round(temp_mean, 2)),
    "humidity"            : int(round(hum_mean, 0)),
    "wind_kph"            : float(round(wind_mean, 2)),
    "air_quality_pm2_5"   : float(round(pm25_mean, 2)),
    "condition_text"      : "Unknown"
})

# Check nulls AFTER filling
print(f"\n📊 Null counts AFTER handling:")
null_after = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).collect()[0].asDict()

total_nulls_after = sum(null_after.values())

if total_nulls_after == 0:
    print(f"   ✅ Zero nulls remaining!")
else:
    for k, v in null_after.items():
        if v > 0:
            print(f"   ⚠️  {k}: {v} nulls remaining")

print(f"\n✅ Null handling complete!")
print(f"   Columns handled : 5")
print(f"   temperature_celsius filled with mean : {round(temp_mean, 2)}")
print(f"   humidity filled with mean            : {int(round(hum_mean, 0))}")
print(f"   wind_kph filled with mean            : {round(wind_mean, 2)}")
print(f"   air_quality_pm2_5 filled with mean   : {round(pm25_mean, 2)}")
print(f"   condition_text filled with           : 'Unknown'")
print(f"   Total nulls before : {total_nulls_before}")
print(f"   Total nulls after  : {total_nulls_after}")

In [0]:
# ── Outlier flagging ───────────────────────────────────────

# Check invalid values BEFORE flagging
print("📊 Invalid values BEFORE flagging:")
print(f"   temperature_celsius > 60  : {df.filter(col('temperature_celsius') > 60).count()} rows")
print(f"   temperature_celsius < -80 : {df.filter(col('temperature_celsius') < -80).count()} rows")
print(f"   humidity > 100            : {df.filter(col('humidity') > 100).count()} rows")
print(f"   humidity < 0              : {df.filter(col('humidity') < 0).count()} rows")
print(f"   uv_index < 0              : {df.filter(col('uv_index') < 0).count()} rows")
print(f"   uv_index > 20             : {df.filter(col('uv_index') > 20).count()} rows")
print(f"   wind_kph > 400            : {df.filter(col('wind_kph') > 400).count()} rows")
print(f"   pressure_mb < 800         : {df.filter(col('pressure_mb') < 800).count()} rows")
print(f"   pressure_mb > 1100        : {df.filter(col('pressure_mb') > 1100).count()} rows")

# Add is_outlier flag
df = df.withColumn(
    "is_outlier",
    when(
        (col("temperature_celsius") > 60)  | (col("temperature_celsius") < -80) |
        (col("humidity") > 100)            | (col("humidity") < 0)              |
        (col("uv_index") < 0)              | (col("uv_index") > 20)             |
        (col("wind_kph") > 400)            |
        (col("pressure_mb") < 800)         | (col("pressure_mb") > 1100),
        True
    ).otherwise(False)
)

outlier_count = df.filter(col("is_outlier") == True).count()
clean_count   = df.filter(col("is_outlier") == False).count()

# Show sample outlier rows
print(f"\n📊 Sample outlier rows:")
display(df.filter(col("is_outlier") == True)
          .select("location_name", "temperature_celsius", "humidity", "uv_index", "wind_kph", "pressure_mb", "is_outlier")
          .limit(5))

print(f"\n✅ Outlier flagging complete!")
print(f"   Columns checked  : 5 (temperature, humidity, uv_index, wind_kph, pressure_mb)")
print(f"   Outliers flagged : {outlier_count}")
print(f"   Clean records    : {clean_count}")
print(f"   NOTE: Outliers are FLAGGED not deleted — Silver keeps all data")

In [0]:
# ── Deduplication ──────────────────────────────────────────

before_count = df.count()

# Show sample duplicates BEFORE removing
print("📊 Sample duplicate rows BEFORE deduplication:")
from pyspark.sql.functions import count as spark_count

dupes = df.groupBy("location_name", "country", "last_updated") \
          .agg(spark_count("*").alias("count")) \
          .filter(col("count") > 1) \
          .orderBy(col("count").desc())

display(dupes.limit(5))
print(f"\n   Total duplicate groups found : {dupes.count()}")

# Remove duplicates
df = df.dropDuplicates(["location_name", "country", "last_updated"])

after_count = df.count()
dupes_removed = before_count - after_count

print(f"\n✅ Deduplication complete!")
print(f"   Columns used as dedup key : 3")
print(f"      location_name")
print(f"      country")
print(f"      last_updated")
print(f"   Rows before : {before_count}")
print(f"   Rows after  : {after_count}")
print(f"   Duplicates removed : {dupes_removed}")

In [0]:
# ── Drop redundant imperial/duplicate columns ──────────────

redundant_cols = [
    "wind_mph",
    "precip_in",
    "visibility_miles",
    "feels_like_fahrenheit",
    "pressure_in",
    "temperature_fahrenheit"
]

cols_before = len(df.columns)

# Show what we're dropping and why
print("📊 Columns being dropped:")
print(f"   wind_mph           → keeping wind_kph (metric standard)")
print(f"   precip_in          → keeping precip_mm (metric standard)")
print(f"   visibility_miles   → keeping visibility_km (metric standard)")
print(f"   feels_like_fahrenheit → keeping feels_like_celsius (metric standard)")
print(f"   pressure_in        → keeping pressure_mb (metric standard)")
print(f"   temperature_fahrenheit → keeping temperature_celsius (metric standard)")

df_silver = df.drop(*redundant_cols)

cols_after = len(df_silver.columns)

print(f"\n✅ Redundant columns dropped!")
print(f"   Columns before : {cols_before}")
print(f"   Columns dropped: {len(redundant_cols)}")
print(f"   Columns after  : {cols_after}")
print(f"\n📋 Final columns in Silver:")
for c in df_silver.columns:
    print(f"   {c}")

In [0]:
# ── Final null check across all columns ───────────────────

print("📊 Final null check across all Silver columns:")

null_counts = df_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).collect()[0].asDict()

cols_with_nulls = {k: v for k, v in null_counts.items() if v > 0}
total_nulls = sum(null_counts.values())

if cols_with_nulls:
    print(f"\n   ⚠️  Columns still with nulls:")
    for k, v in cols_with_nulls.items():
        print(f"      {k}: {v} nulls")
else:
    print(f"\n   ✅ Zero nulls across all {len(df_silver.columns)} columns!")

print(f"\n✅ Final null check complete!")
print(f"   Total columns checked : {len(df_silver.columns)}")
print(f"   Columns with nulls    : {len(cols_with_nulls)}")
print(f"   Total nulls remaining : {total_nulls}")

In [0]:
# ── Add Silver audit columns ───────────────────────────────
# Audit columns are metadata columns that track WHERE the data came from,
# WHEN it was processed, and WHICH layer it belongs to.
# Every layer (Bronze, Silver, Gold) adds its own audit columns
# so you can trace any record back through the entire pipeline.

# Count columns before adding audit cols — so we can show how many we added
cols_before = len(df_silver.columns)

df_silver = (
    df_silver
    # Today's date as a string — tells you WHICH RUN created this data
    # Useful when you run the pipeline daily — you can filter by batch
    .withColumn("_batch_date",   lit(batch_date))

    # Exact timestamp when this record was written to Silver
    # Helps in debugging — "when exactly did this record get processed?"
    .withColumn("_ingestion_ts", current_timestamp())

    # Where did Silver read the data FROM — always bronze_delta/ for Silver
    # Creates a full data lineage trail: raw → bronze_delta → silver
    .withColumn("_source_path",  lit(f"{BUCKET}/bronze_delta/"))

    # Tags every record with which layer it belongs to
    # When Gold reads Silver, it can confirm data came from correct layer
    .withColumn("_layer",        lit("silver"))
)

# Count columns after
cols_after = len(df_silver.columns)

# Show the actual values in audit columns for 3 sample rows
print("📊 Audit columns added:")
print(f"   _batch_date   : {batch_date}  ← today's date")
print(f"   _ingestion_ts : current timestamp ← exact processing time")
print(f"   _source_path  : {BUCKET}/bronze_delta/ ← where data came from")
print(f"   _layer        : silver ← which layer produced this")

display(df_silver.select("_batch_date", "_ingestion_ts", "_source_path", "_layer").limit(3))

print(f"\n✅ Audit columns added!")
print(f"   Columns before : {cols_before}")
print(f"   Columns added  : {cols_after - cols_before}")
print(f"   Columns after  : {cols_after}")

In [0]:
# ── Write Silver Delta to S3 ───────────────────────────────
# This is the actual save step — everything before this was just
# transformations in memory. This cell physically writes the
# cleaned data to S3 as a Delta table.

print(f"📊 Writing Silver Delta to S3...")
print(f"   Target path    : {SILVER_PATH}")
print(f"   Format         : Delta")
print(f"   Mode           : Overwrite (replaces previous Silver data)")
print(f"   Partition by   : _batch_date (one folder per daily run)")
print(f"   Total rows     : {df_silver.count()}")
print(f"   Total columns  : {len(df_silver.columns)}")

# Write to S3 as Delta format
# mode overwrite → replaces existing Silver data on every run (no duplicates accumulate)
# overwriteSchema → allows schema changes between runs without errors
# partitionBy _batch_date → creates subfolders by date for faster querying
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("_batch_date") \
    .save(SILVER_PATH)

print(f"\n✅ Silver Delta written successfully!")
print(f"   Location : {SILVER_PATH}")

In [0]:
# ── Register Silver table in Unity Catalog ─────────────────
# Writing to S3 above only saves the files physically.
# This step registers it as a proper queryable table in Unity Catalog
# so you can run SQL like: SELECT * FROM global_weather_catalog.silver.weather_silver
# Without this step, the data exists in S3 but nobody can query it by name.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_silver
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print(f"✅ Silver table registered in Unity Catalog!")
print(f"   Catalog  : {CATALOG}")
print(f"   Schema   : silver")
print(f"   Table    : weather_silver")
print(f"   Location : {SILVER_PATH}")
print(f"   Format   : Delta")
print(f"\n   You can now query it with:")
print(f"   SELECT * FROM {CATALOG}.silver.weather_silver")

In [0]:
%sql
SELECT * FROM global_weather_catalog.silver.weather_silver

In [0]:
# ── Optimize Silver table ──────────────────────────────────
# OPTIMIZE compacts all the small Delta files written to S3
# into fewer, larger files — making queries much faster.
# ZORDER BY country, last_updated means if someone queries
# "WHERE country = 'India'" it scans far fewer files.

spark.sql(f"OPTIMIZE {CATALOG}.silver.weather_silver ZORDER BY (country, last_updated)")

print(f"✅ OPTIMIZE complete!")
print(f"   Table    : {CATALOG}.silver.weather_silver")
print(f"   ZORDER BY: country, last_updated")
print(f"\n   What this did:")
print(f"   → Compacted small S3 files into larger ones")
print(f"   → Co-located data by country + last_updated")
print(f"   → Queries filtering by country will now be much faster")

In [0]:
# ── Final verification ─────────────────────────────────────
# This is the final sanity check — confirms everything
# landed correctly in the Silver table in Unity Catalog.
# We check row count, outlier count, null count and sample data.

written = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {CATALOG}.silver.weather_silver"
).collect()[0]["cnt"]

outliers = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {CATALOG}.silver.weather_silver WHERE is_outlier = true"
).collect()[0]["cnt"]

clean = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {CATALOG}.silver.weather_silver WHERE is_outlier = false"
).collect()[0]["cnt"]

null_check = spark.sql(f"""
    SELECT
        SUM(CASE WHEN temperature_celsius IS NULL THEN 1 ELSE 0 END) AS temp_nulls,
        SUM(CASE WHEN humidity IS NULL THEN 1 ELSE 0 END)            AS humidity_nulls,
        SUM(CASE WHEN wind_kph IS NULL THEN 1 ELSE 0 END)            AS wind_nulls,
        SUM(CASE WHEN air_quality_pm2_5 IS NULL THEN 1 ELSE 0 END)   AS pm25_nulls,
        SUM(CASE WHEN condition_text IS NULL THEN 1 ELSE 0 END)      AS condition_nulls
    FROM {CATALOG}.silver.weather_silver
""").collect()[0]

print(f"🎉 Silver Layer — Final Verification Report")
print(f"{'='*50}")
print(f"\n📊 Row Counts:")
print(f"   Total rows written  : {written}")
print(f"   Clean records       : {clean}")
print(f"   Outliers flagged    : {outliers}")
print(f"\n📊 Null Check (should all be 0):")
print(f"   temperature_celsius : {null_check['temp_nulls']}")
print(f"   humidity            : {null_check['humidity_nulls']}")
print(f"   wind_kph            : {null_check['wind_nulls']}")
print(f"   air_quality_pm2_5   : {null_check['pm25_nulls']}")
print(f"   condition_text      : {null_check['condition_nulls']}")
print(f"\n📊 Silver Summary:")
print(f"   Catalog  : {CATALOG}")
print(f"   Schema   : silver")
print(f"   Table    : weather_silver")
print(f"   Location : {SILVER_PATH}")
print(f"   Format   : Delta")
print(f"   Columns  : 60")
print(f"\n✅ Silver ingestion SUCCESSFUL ✓")

# Show sample rows
display(spark.sql(f"SELECT * FROM {CATALOG}.silver.weather_silver LIMIT 5"))